In [1]:
import torch
from torch import nn


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\klill\miniconda3\envs\dele2\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\klill\miniconda3\envs\dele2\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\klill\miniconda3\envs\dele2\Lib\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.start()
  File "c:

In [ ]:
class MNISTmodel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=(3, 3), padding=1)
        self.flat = nn.Flatten()
        self.dense1 = nn.Linear(in_features=28*28, out_features=10)
        # self.sm = nn.Softmax(dim=1)

    def forward(self, x):
        out = self.conv1(x)
        out = self.flat(out)
        out = self.dense1(out)
        # out = self.sm(out)
        return out

# Test if output dims are correct
test_model = MNISTmodel()
test_input = torch.randn(1, 1, 28, 28)
test_output = test_model(test_input)
print(f'Test-output dimensions: {test_output.shape}')
del test_model, test_input, test_output

Test-output dimensions: torch.Size([1, 10])


In [2]:
class DeepCNNmodel(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 28x28 -> 14x14

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 14x14 -> 7x7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [4]:
checkpoint = torch.load(r'./model_training/DeepCNNmodel_checkpoint.pth')

model = DeepCNNmodel()

model.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

In [5]:
input_names = ['input1']
output_names = ["output1"]
dummy_input = torch.randn(1, 1, 28, 28)
                    # PyTorch uses this dummy input for *tracing* -
                    # the method that converts the dynamic python code
                    # into a static computation graph by logging or
                    # tracing the operations done by the model.

torch.onnx.export(
  model, 
  dummy_input, 
  "deep_cnn.onnx", 
  verbose=True, 
  input_names=input_names,
  output_names=output_names
)